# Exercise 3: Earthquake Magnitude-Frequency Analysis Near Antananarivo

This notebook demonstrates how to analyze the frequency and magnitude of earthquakes near Antananarivo, Madagascar, using real earthquake catalog data. The workflow includes:

- **Data Acquisition:** Earthquake events are fetched from the USGS catalog for a region surrounding Antananarivo, defined by latitude and longitude bounds, and filtered by minimum magnitude.
- **Data Cleaning:** Events with missing depth or magnitude information are investigated to get the most preferred values.
- **Distance Calculation:** The geographic distance from each earthquake to Antananarivo is calculated by solving the inverse geodesic problem using Karney’s high-precision geodesic algorithms.
- **Magnitude Binning:** Earthquake magnitudes are grouped into bins (0.5 increments), and the number of events in each bin is counted.
- **Magnitude-Frequency Distribution:** The cumulative number of events above each magnitude threshold is calculated and visualized, showing the typical decrease in frequency with increasing magnitude.
- **Gutenberg-Richter Law Fitting:** The Gutenberg-Richter relation (`log10(N) = a - b*M`) is fitted to the magnitude-frequency data, providing estimates for the parameters `a` and `b` that characterize the seismicity of the region.
- **Data Filtering:** The analysis is repeated for events with magnitude ≥ 4.0 to focus on more significant earthquakes.
- **Apply Truncated Gutenberg-Richter**: Apply the truncated Gutenberg-Richter (TGR) model $ N(M) = K \left(10^{-bM} - 10^{-bM_{\max}}\right) $ accounts for physical magnitude limits by introducing a sharp cutoff at $M_{\max}$. 

This exercise illustrates key concepts in seismology and statistical analysis, including catalog querying, data wrangling, binning, cumulative distributions, and linear regression fitting for empirical laws. The workflow is modular and can be adapted to other regions or magnitude thresholds.

First, we need to install the required packages.

In [ ]:
# 1) Install dependencies if missing
import sys, subprocess, importlib

def ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
        print(f"[ok] {pkg} already installed")
    except Exception:
        print(f"[info] Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        importlib.import_module(name)
        print(f"[ok] {pkg} installed")


pkgs = ["os", "re", "requests", "zipfile", "geopandas",
        "matplotlib", "shapely", "cartopy", "obspy", 
        "folium", "osmnx", "numpy", "pandas", "geographiclib"]

for pkg in pkgs:
    ensure(pkg)

# Import required libraries

In the next cell, we will import the required packages and define the parameters and constants needed for the exercise.

In [ ]:
#%% Imports and constants
import numpy as np
import matplotlib.pyplot as plt
from math import radians, sin, cos, asin, sqrt, log10, log, exp
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
import geopandas as gpd
import requests, zipfile, io
from obspy.geodetics import gps2dist_azimuth
import pandas as pd
from copy import deepcopy

# Constants
R_EARTH_KM = 6371.0  # Earth radius in kilometers

# City of interest
CITY_NAME = "Antananarivo"
CITY_LAT = -18.9
CITY_LON = 47.5

# Earthquake catalog search parameters
MIN_LAT = -26.0
MAX_LAT = -11.0
MIN_LON = 42.0
MAX_LON = 51.0
START_DATE = UTCDateTime(1900, 1, 1)
END_DATE = UTCDateTime.now()
MIN_MAG = 3.0

# Fetching and Cleaning Earthquake Catalog Data

In this step, we query the USGS earthquake catalog for events within the specified latitude and longitude bounds around Antananarivo, Madagascar, and within the defined time and magnitude range. We then filter out events that lack depth or magnitude information to ensure the dataset is suitable for further analysis. The resulting catalog contains only earthquakes with complete and reliable data.

In [ ]:
#%% Fetch earthquake catalog using ObsPy
client = Client("USGS")
catalog = client.get_events(starttime=START_DATE, endtime=END_DATE,
                            minlatitude=MIN_LAT, maxlatitude=MAX_LAT,
                            minlongitude=MIN_LON, maxlongitude=MAX_LON,
                            minmagnitude=MIN_MAG)
print(f"Fetched {catalog.count()} earthquake events from the catalog")

# Check events with no magnitude or depth and use the preferred origin/magnitude if available
clean_events = []

for event in catalog:
    origin = event.preferred_origin() or (event.origins[0] if event.origins else None)
    magnitude = event.preferred_magnitude() or (event.magnitudes[0] if event.magnitudes else None)

    if origin is None or magnitude is None:
        continue

    if origin.depth is None or magnitude.mag is None:
        continue

    clean_events.append(event)

print(f"Original catalog size: {catalog.count()}")
print(f"Cleaned catalog size: {len(clean_events)}")

# Calculating Distances and Preparing Event Data

In this step, we calculate the distance from each earthquake event to Antananarivo using the haversine formula, which accounts for the curvature of the Earth. We then extract relevant event attributes—latitude, longitude, depth, magnitude, and computed distance—and organize them into a GeoDataFrame for spatial analysis. This structured dataset enables further statistical and geographic exploration of earthquake patterns near Antananarivo.

In [ ]:
#%% Extract event data (preferred origin/magnitude) and compute distances
# NOTE:
# Here we compute point-source distances only:
#   - Repi: epicentral (surface) distance
#   - Rhyp: hypocentral distance (uses depth)
# These are NOT the same as rupture-based GMPE distances (Rjb, Rrup),
# which require rupture geometry (fault/rupture surface). If you apply a GMPE later,
# compute the distance metric required by that GMPE.

event_data = []

for event in clean_events:  # use the cleaned list from the previous step
    origin = event.preferred_origin() or (event.origins[0] if event.origins else None)
    magobj = event.preferred_magnitude() or (event.magnitudes[0] if event.magnitudes else None)

    if origin is None or magobj is None or origin.depth is None or magobj.mag is None:
        continue

    # More robust than manual haversine (already imported in your notebook)
    dist_m, az, baz = gps2dist_azimuth(CITY_LAT, CITY_LON, origin.latitude, origin.longitude)
    repi_km = dist_m / 1000.0
    depth_km = origin.depth / 1000.0
    rhyp_km = np.sqrt(repi_km**2 + depth_km**2)

    event_data.append({
        "latitude": origin.latitude,
        "longitude": origin.longitude,
        "depth_km": depth_km,
        "magnitude": magobj.mag,
        "Repi_km": repi_km,
        "Rhyp_km": rhyp_km
    })

events_gdf = gpd.GeoDataFrame(
    event_data,
    geometry=gpd.points_from_xy([e["longitude"] for e in event_data],
                                [e["latitude"] for e in event_data]),
    crs="EPSG:4326"
)

events_gdf_raw = deepcopy(events_gdf)

print(f"Prepared GeoDataFrame with {len(events_gdf)} events")

# Magnitude Binning and Frequency Analysis

In this section, earthquake events are grouped into magnitude bins with 0.5 increments. The number of events in each bin is counted, and a cumulative count is calculated to show how many earthquakes have magnitudes greater than or equal to each bin threshold. This cumulative distribution is a key step in understanding the frequency of larger earthquakes in the region. The results are visualized using a histogram and a magnitude-frequency scatter plot, which reveal the typical decrease in event frequency with increasing magnitude.

In [ ]:
#%% Classify events by magnitude into bins with 0.5 increments
# and get the count of events in each bin

bins = np.arange(MIN_MAG, max(events_gdf['magnitude']) + 0.5, 0.5)
events_gdf['mag_bin'] = pd.cut(events_gdf['magnitude'], bins)
mag_bin_counts = events_gdf['mag_bin'].value_counts().sort_index()
mag_bins_centers = bins[:-1] + 0.25  # center of each bin


# Make a new data frame for better with the bins, number of events, bin centers,
# and cumulative counts from the largest to smallest magnitude
mag_bin_df = pd.DataFrame({
    'magnitude_bin': mag_bin_counts.index.astype(str),
    'event_count': mag_bin_counts.values,
    'bin_center': bins[:-1] + 0.25  # center of each bin
})
cum_counts = mag_bin_df['event_count'][::-1].cumsum()[::-1]
mag_bin_df['cumulative_count'] = cum_counts.values

# Add a column for log10 of cumulative counts, replacing 0 with NaN to avoid -inf
# Calculate log10 of cumulative counts, handling zero counts
log_counts = lambda x: log10(x) if x > 0 else np.nan
mag_bin_df['log_counts'] = mag_bin_df['cumulative_count'].replace(0, np.nan).apply(log_counts)
print(mag_bin_df)

# Plot histogram of event counts by magnitude bins
plt.figure(figsize=(10, 6))
mag_bin_counts.plot(kind='bar', color='skyblue', edgecolor='black')
plt.xlabel("Magnitude Bins")
plt.ylabel("Number of Events")
plt.title("Histogram of Earthquake Magnitudes")
plt.grid(axis='y')
plt.show()

#%% Calculate and plot magnitude-frequency distribution
# Plot magnitude-frequency distribution
plt.figure(figsize=(10, 6))
plt.scatter(mag_bins_centers, cum_counts, color='red')
plt.yscale('log')
plt.xlabel("Magnitude")
plt.ylabel("Cumulative Number of Events (log scale)")
plt.title("Magnitude-Frequency Distribution")
plt.grid()
plt.show()


# Fitting the Gutenberg-Richter Law

In this step, we fit the Gutenberg-Richter relation to the magnitude-frequency data. The Gutenberg-Richter law describes the relationship between earthquake magnitude and the cumulative number of events, expressed as `log10(N) = a - b*M`, where `N` is the cumulative count and `M` is magnitude. By performing linear regression on the log-transformed cumulative counts versus magnitude bin centers, we estimate the parameters `a` and `b`, which characterize the seismicity of the region. The observed data and the fitted model are visualized together to assess the fit quality.

In [ ]:
#%% Fit Gutenberg-Richter relation to the magnitude-frequency data
# Gutenberg-Richter relation: log10(N) = a - b*M
# where N is the cumulative number of events with magnitude >= M
# We can fit a linear model to log10(N) vs M to estimate a and b
from scipy.stats import linregress

log_counts = np.log10(cum_counts)
slope, intercept, r_value, p_value, std_err = linregress(mag_bins_centers, log_counts)
b_value = -slope
a_value = intercept 
print(f"Fitted Gutenberg-Richter parameters: a = {a_value:.2f}, b = {b_value:.2f}")
# Plot fitted line
plt.figure(figsize=(10, 6))
plt.scatter(mag_bins_centers, cum_counts, color='red', label='Observed Data')
plt.yscale('log')
plt.xlim((MIN_MAG, max(bins)))
plt.xlabel("Magnitude")
plt.ylabel("Cumulative Number of Events (log scale)")
plt.title("Magnitude-Frequency Distribution with Gutenberg-Richter Fit")
plt.grid()
# Plot fitted line
fitted_counts = 10**(a_value - b_value * mag_bins_centers)
plt.plot(mag_bins_centers, fitted_counts, color='blue', label='Gutenberg-Richter Fit')
plt.legend()
plt.show()

## Filtering for Significant Earthquakes (Magnitude ≥ 4.0)

We will focus the analysis on a more complete record and impactful seismic events, we filter the earthquake catalog to include only those with magnitude greater than or equal to 4.0. Events with magnitude less than 4.0 are fewer and missing, which does not match the general observation that the number of earthquakes increases as the magnitude decreases. This is mainly related to the lack or limited number of seismological stations in the region. This step reduces the influence of smaller, less significant earthquakes and allows for a clearer examination of the magnitude-frequency relationship among moderate and large events. The filtered dataset is used to recalculate magnitude bins, cumulative counts, and to visualize the updated magnitude-frequency distribution. This targeted approach provides more meaningful insights into the seismic hazard near Antananarivo.

In [ ]:
#%% Remove events with magnitude less than 4.0
events_gdf = mag_bin_df[mag_bin_df['bin_center'] >= 4.0]
print(events_gdf)

# Recalculate magnitude bins and cumulative counts
# Plot magnitude-frequency distribution
plt.figure(figsize=(10, 6))
plt.scatter(events_gdf['bin_center'], events_gdf['cumulative_count'], color='red')
plt.yscale('log')
plt.xlabel("Magnitude")
plt.xlim((4.0, max(bins)))
plt.ylabel("Cumulative Number of Events (log scale)")
plt.title("Magnitude-Frequency Distribution")
plt.grid()
plt.show()

# Gutenberg-Richter Law Fit for Significant Earthquakes

In this step, we fit the Gutenberg-Richter relation to the filtered magnitude-frequency data (magnitude ≥ 4.0). The linear regression is performed on the log-transformed cumulative event counts versus the magnitude bin centers. The resulting parameters `a` and `b` quantify the seismicity for moderate and large earthquakes near Antananarivo. The plot displays both the observed cumulative counts and the fitted Gutenberg-Richter model, allowing visual assessment of the fit and interpretation of regional earthquake occurrence rates.

In [ ]:
# %% Fit Gutenberg-Richter relation to the new magnitude-frequency data
# Gutenberg-Richter relation: log10(N) = a - b*M
# where N is the cumulative number of events with magnitude >= M
# We can fit a linear model to log10(N) vs M to estimate a and b
from scipy.stats import linregress

slope, intercept, r_value, p_value, std_err = linregress(events_gdf['bin_center'], events_gdf['log_counts'])
b_value = -slope
a_value = intercept 
print(f"Fitted Gutenberg-Richter parameters after cleaning: a = {a_value:.2f}, b = {b_value:.2f}")
# Plot fitted line
plt.figure(figsize=(10, 6))
plt.scatter(events_gdf['bin_center'], events_gdf['cumulative_count'], color='red', label='Observed Data')
plt.yscale('log')
plt.xlabel("Magnitude")
plt.xlim((4.0, max(bins)))
plt.ylabel("Cumulative Number of Events (log scale)")
plt.title("Magnitude-Frequency Distribution with Gutenberg-Richter Fit")
plt.grid()
# Plot fitted line
fitted_counts = 10**(a_value - b_value * mag_bins_centers)
plt.plot(mag_bins_centers, fitted_counts, color='blue', label='Gutenberg-Richter Fit')
plt.legend()
plt.show()

## The Truncated Gutenberg–Richter (TGR) Relationship

The standard Gutenberg-Richter law ($log_{10}N = a - bM$) implies that earthquakes can occur at infinitely large magnitudes, which is physically impossible due to the finite size of tectonic plates and faults. The **Truncated Gutenberg-Richter (TGR)** model introduces a sharp cutoff at a maximum magnitude $M_{max}$.

### The Mathematical Model

The cumulative number of earthquakes $N$ with a magnitude greater than or equal to $M$ is expressed as:

$$N(M) = K \cdot (10^{-bM} - 10^{-bM_{max}})$$

Where:
*   **$M$**: The magnitude of interest.
*   **$b$**: The $b$-value, representing the relative size distribution (slope).
*   **$M_{max}$**: The maximum possible magnitude the region can support.
*   **$K$**: A constant related to the total seismicity rate (similar to $10^a$ in the standard law).

### Implementation Details
In our Python implementation using `scipy.optimize.curve_fit`, we use this non-linear form to better capture the 'curvature' or 'rollover' seen in the data as magnitudes approach the physical limit of the region. Unlike the linear regression used for the standard law, this approach treats $M_{max}$ as a parameter to be optimized within physically realistic bounds.

We define the initial guess as $ M_{\max} = \max(M_{\text{data}}) + 0.5 $ assuming the catalog is missing the next magnitude step for the non-linear optimizer.


In [ ]:
# The Truncated Gutenberg-Richter law implementation 
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Focus on the same filtered data used for standard GR (Magnitude >= 4.0)
# This is more reliable for fitting seismic laws.
fitting_df = events_gdf.dropna(subset=['log_counts'])

M_data = fitting_df['bin_center'].values
N_data = fitting_df['cumulative_count'].values

def truncated_gr_cumulative_N(M, K, b, M_max):
    result = np.zeros_like(M, dtype=float)
    valid_indices = M < M_max
    if np.any(valid_indices):
        # N(M) = K * (10^-bM - 10^-bMmax)
        term_M = 10**(-b * M[valid_indices])
        term_Mmax = 10**(-b * M_max)
        val = K * (term_M - term_Mmax)
        result[valid_indices] = np.maximum(1e-10, val) # Avoid 0 for log scaling
    return result

# 1. Improved Initial Guesses
# K can be estimated from N(M_min)
K_guess = N_data[0] / (10**(-b_value * M_data[0])) if 'b_value' in locals() else 10**5
b_guess = b_value if 'b_value' in locals() else 1.0
M_max_guess = np.max(M_data) + 0.5 

# 2. Realistic Physical Bounds
# M_max should not exceed ~9.5 (largest ever recorded on Earth)
# b-value typically ranges between 0.6 and 1.5
lower_bounds = [1.0, 0.6, np.max(M_data) + 0.1]
upper_bounds = [1e9, 1.5, 9.5]

try:
    params, covariance = curve_fit(truncated_gr_cumulative_N, M_data, N_data,
                                   p0=[K_guess, b_guess, M_max_guess],
                                   bounds=(lower_bounds, upper_bounds))
    K_fit, b_fit, M_max_fit = params

    print(f"Fitted Truncated G-R parameters (constrained):")
    print(f"  K     = {K_fit:.2f}")
    print(f"  b     = {b_fit:.2f} (Seismologically realistic range: 0.6-1.5)")
    print(f"  M_max = {M_max_fit:.2f} (Constrained to <= 9.5)")

    plt.figure(figsize=(10, 6))
    plt.scatter(M_data, N_data, color='red', label='Observed Data (M >= 4)')
    plt.yscale('log')
    
    # Generate smooth curve
    M_plot = np.linspace(4.0, M_max_fit, 100)
    N_plot = truncated_gr_cumulative_N(M_plot, K_fit, b_fit, M_max_fit)
    
    plt.plot(M_plot, N_plot, color='green', linewidth=2, 
             label=f'Truncated G-R Fit (b={b_fit:.2f}, Mmax={M_max_fit:.1f})')
    
    plt.xlabel("Magnitude")
    plt.ylabel("Cumulative Number of Events (log scale)")
    plt.title("Magnitude-Frequency: Truncated G-R with Physical Constraints")
    plt.legend()
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.show()

except Exception as e:
    print(f"Fit failed: {e}")

# Extra tasks:
* Did you observe variation in the a and b parameters based on the input? Why does this variation occur?
* What is the effect of binning? Try redoing the analysis with different bins, e.g., with a 0.2 interval.
* Did you observe variation in the a and b estimates?
* What does this tell you about the importance of the completeness of earthquake catalogs?
* Are you aware of other fitting approaches? Try them. For example, you could use the numpy.polyfit function.
* Is there any difference between the a and b estimates using numpy.polyfit and the scipy linregress function used here?
* In your opinion, what is more important: the choice of bins or the fitting function?
